<a href="https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayush13007/flyrankweek1assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My lane is Content Refresh Prioritization (predicting content decay). This is a Binary Classification task.

We need to predict whether a specific piece of content is currently declining and requires a refresh (Class 1) or is stable/growing (Class 0). Classification is the right fit because the final business action is binary: route this URL to an editor to update it, or leave it alone.

In [ ]:
import pandas as pd
import numpy as np

# Setting up pandas to view our data clearly without truncating columns
pd.set_option('display.max_columns', 50)
print("Environment ready for binary classification task.")

Environment ready for binary classification task.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Our target is the is_declining_label column.

This is a defined rule (a proxy) derived from trend_direction and trend_pct. Because the label is strictly derived from these two columns, they contain the answer key. To prevent data leakage (a massive gotcha in this dataset), we must strictly drop trend_pct and trend_direction from our feature set before any modeling.

In [ ]:
# Define our target variable and the leaky features we MUST drop
target_col = 'is_declining_label'
leaky_cols = ['trend_pct', 'trend_direction']

print(f"Target to predict: {target_col}")
print(f"Leakage columns to remove before training: {leaky_cols}")

Target to predict: is_declining_label
Leakage columns to remove before training: ['trend_pct', 'trend_direction']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Our success metric should be Precision and ROC-AUC.

Why Precision? The output supports human editors who have limited time. If the model tells them to refresh a post (predicts True), it needs to be right. A high False Positive rate wastes expensive human editorial hours on content that is actually fine. We want a model that is highly confident when it flags a post for review.

In [ ]:
from sklearn.metrics import precision_score, roc_auc_score

# Dummy check to represent our metric logic
y_true_mock = [1, 0, 1, 1, 0]
y_pred_mock = [1, 0, 0, 1, 0]

print(f"Primary Metric: Precision (Mock calculation: {precision_score(y_true_mock, y_pred_mock):.2f})")
print("Action supported: Sending high-precision 'Needs Refresh' lists to content teams.")

Primary Metric: Precision (Mock calculation: 1.00)
Action supported: Sending high-precision 'Needs Refresh' lists to content teams.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The unit of analysis is one row = one pseudonymized content item (URL/post) for a specific client over a trailing 90-day window.

In [ ]:
import os
import pandas as pd

# If running in Google Colab and repo isn't cloned yet, clone it
repo_name = 'flyrankweek1assignment'
if not os.path.exists('data') and not os.path.exists(f'../data'):
    if not os.path.exists(repo_name):
        !git clone https://github.com/ayush13007/flyrankweek1assignment.git
    os.chdir(repo_name)

# Candidate paths for the CSV
possible_paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    f'{repo_name}/data/raw/content_refresh_anonymized.csv'
]

file_path = None
for path in possible_paths:
    if os.path.exists(path):
        file_path = path
        break

if file_path is None:
    raise FileNotFoundError("Could not locate 'content_refresh_anonymized.csv'. Ensure the repo is accessible.")

# 1. Load the raw dataset
df = pd.read_csv(file_path)

# 2. Engineer the target proxy BEFORE dropping the leaky columns
target_col = 'is_declining_label'
if 'trend_pct' in df.columns:
    # We define "declining" as a negative trend percentage
    df[target_col] = (df['trend_pct'] < 0).astype(int)
else:
    # Fallback just in case the column names differ slightly
    df[target_col] = 0

# 3. Drop the leaky columns so the model can't cheat
leaky_cols = ['trend_pct', 'trend_direction']
df_clean = df.drop(columns=leaky_cols, errors='ignore')

print(f"Success! Dataset loaded from: {file_path}")
print(f"Shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print("Unit of analysis: 1 row = 1 piece of content (content_id)\n")

# Safely display the columns to prove our unit of analysis
display_cols = [c for c in ['client_id', 'content_id', 'content_type', target_col] if c in df_clean.columns]
display(df_clean[display_cols].head())

Success! Dataset loaded from: data/raw/content_refresh_anonymized.csv
Shape: 30000 rows x 43 columns
Unit of analysis: 1 row = 1 piece of content (content_id)



,client_id,content_id,content_type,is_declining_label
0,client_f369cb89fc,content_304f48230142,keyword article,1
1,client_4e07408562,content_a1fb4e703a9e,keyword article,1
2,client_7f2253d7e2,content_9aa793d4d895,keyword article,1
3,client_19581e27de,content_331d6c4de07b,keyword article,1
4,client_3fdba35f04,content_d99b7a2d90ca,keyword article,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


A fixed rule (e.g., IF traffic drops > 10% AND age > 1 year THEN refresh) is too rigid and reactive.

ML beats a fixed rule because content decay is multi-dimensional. A 3000-word guide with high scroll_rate but dropping search positions behaves differently than a short news brief with high ai_traffic_pct. An ML model can weigh 40+ subtle, interactive signals simultaneously—accounting for differing missingness patterns (like missing keyword data by content type) without failing, catching nuanced decay patterns before a simple threshold would trigger.

In [ ]:
# Demonstrating the complexity: A fixed rule only looks at 1-2 dimensions.
# ML will look at all of these simultaneously.

features_to_weigh = [
    'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'word_count', 'avg_position'
]

print("An IF/THEN rule struggles to balance these interacting features:")

# Verify df_clean exists in memory before running summary statistics
if 'df_clean' in globals() or 'df_clean' in locals():
    # Only try to display features that actually exist in the dataframe
    available_features = [f for f in features_to_weigh if f in df_clean.columns]

    if available_features:
        display(df_clean[available_features].describe().T[['mean', 'min', 'max']])
    else:
        print("None of the specified features were found in the dataframe.")
else:
    print("Error: 'df_clean' is not defined. Please run Cell 4 first to load the data.")

An IF/THEN rule struggles to balance these interacting features:


,mean,min,max
ctr,0.510733,0.0,100.0
engagement_rate,2.534520,0.0,100.0
scroll_rate,18.212921,0.0,300.0
ai_traffic_pct,0.768196,0.0,300.0
word_count,3107.760325,8.0,9546.0
avg_position,16.342380,0.0,245.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.